# 03 -- Paragraph-level TCM, support floor, K=9 sub-space

Paragraph level takes the units to be the paragraphs produced by the
detectors' own segmentation; gold and predicted character spans are assigned
to every paragraph they overlap (paper §III-C).

Two things happen here:
1. the **support-floor sweep** of Table I;
2. the **K=9 common sub-space** on which every paragraph-level statistic in
   the paper is computed.

The floor is not cosmetic. Bi-normalisation rescales near-empty rows without
bound, and notebook 07 shows what that does when the floor is removed.

Expected: K=9 at s=1, all nine configurations converging within 26 iterations,
peak off-diagonal weight 0.40.

In [ ]:
import sys; sys.path.insert(0, 'src')
from common import *

PARA_IDX = {l: build_para_index(l) for l in LANGS}
for l in LANGS:
    assert sum(len(v) for v in PARA_IDX[l].values()) == EXPECT_PARAGRAPHS[l]

GOLD_PS, PRED_PS, KEYS = {}, {}, {}
for l in LANGS:
    GOLD_PS[l], _ = spans_to_paragraphs(load_gold(l), PARA_IDX[l])
    KEYS[l] = paragraph_keys(PARA_IDX[l])
    for m in METHODS:
        PRED_PS[(l, m)], _ = spans_to_paragraphs(load_pred(l, m), PARA_IDX[l])
print('paragraph-level gold and predictions mapped')

In [ ]:
# --- nine raw paragraph-level TCMs (full 23x23) ---
RAW_PAR, NUSED = {}, {}
print(f"{'config':<18} {'units used':>11} {'available':>10} {'diag':>7}")
print('-' * 50)
for l, m in CONFIGS:
    Yg, Yp = paragraph_multihot(GOLD_PS[l], PRED_PS[(l, m)], KEYS[l])
    T, n = compute_tcm(Yg, Yp)
    RAW_PAR[(l, m)], NUSED[(l, m)] = T, n
    np.save(MATRICES / 'paragraph' / f'{l}_{m}_raw.npy', T)
    print(f'{LANG_LABEL[l]} {METHOD_LABEL[m]:<12} {n:>11} '
          f'{len(KEYS[l]):>10} {np.trace(T)/T.sum():>6.1%}')

print('\nPaper §IV-E: the TCM sees only units where gold and predictions are')
print('both non-empty -- 818/3,127 for EN Sup-FT, 108/503 for RU Prompt-A.')
print('Paper §IV-A: paragraph-level diagonal share is 38-49%.')
pd.DataFrame([{'lang': l, 'method': m, 'n_used': NUSED[(l, m)],
               'n_available': len(KEYS[l])} for l, m in CONFIGS]
             ).to_csv(RESULTS / 'paragraph_units_used.csv', index=False)

In [ ]:
# --- Table I: support-floor sweep ---
rows = []
for s in [0.5, 1, 2, 3, 5, 8, 10]:
    keeps = [support_floor_keep(T, s) for T in RAW_PAR.values()]
    common = sorted(set.intersection(*keeps), key=lambda t: TECH2IDX[t])
    K = len(common)
    if K < 3:
        rows.append({'s': s, 'K': K, 'pairs': None,
                     'all_converged': None, 'max_iters': None})
        continue
    idx = [TECH2IDX[t] for t in common]
    allconv, maxit = True, 0
    for T in RAW_PAR.values():
        _, it, cv = bi_norm(T[np.ix_(idx, idx)])
        allconv &= cv
        maxit = max(maxit, it)
    rows.append({'s': s, 'K': K, 'pairs': K * (K - 1) // 2,
                 'all_converged': allconv, 'max_iters': maxit})

sweep = pd.DataFrame(rows)
sweep.to_csv(RESULTS / 'table1_supsweep.csv', index=False)
print(sweep.to_string(index=False))
print('\nWe adopt s = 1: largest K that converges quickly. At s = 0.5 one')
print('further technique is retained but IPF needs ~3,378 iterations,')
print('indicating residual ill-conditioning (paper §III-B).')

In [ ]:
# --- the K=9 common sub-space ---
COMMON, IDX = common_subspace(RAW_PAR, MIN_SUP)
K = len(COMMON)
assert K == 9, f'expected K=9 at s={MIN_SUP}, got {K}'

print(f'K = {K}\n')
print(f"{'abbr':<6} {'super-class':<8} {'technique'}")
print('-' * 52)
for t in COMMON:
    print(f'{ABBR[t]:<6} {TECH2SUPER[t]:<8} {t}')

from collections import Counter
print(f'\nsuper-class distribution: {dict(Counter(TECH2SUPER[t] for t in COMMON))}')
print('(paper Limitations: uneven -- six Pathos, two Ethos, one Logos)')
print(f'\nexcluded ({C-K}): {sorted(set(TECHNIQUES) - set(COMMON))}')
pd.DataFrame({'technique': COMMON,
              'abbr': [ABBR[t] for t in COMMON],
              'superclass': [TECH2SUPER[t] for t in COMMON]}
             ).to_csv(RESULTS / 'table2_common_techniques.csv', index=False)

In [ ]:
# --- bi-normalise on the sub-space; all nine must converge ---
BI_PAR = {}
print(f"{'config':<18} {'IPF':>5} {'iters':>7} {'max off-diag':>13}")
print('-' * 48)
peak = 0.0
for l, m in CONFIGS:
    Tb, it, cv = bi_norm(RAW_PAR[(l, m)][np.ix_(IDX, IDX)])
    assert cv, f'{l}/{m}: IPF did not converge on the K=9 sub-space'
    BI_PAR[(l, m)] = Tb
    np.save(MATRICES / 'paragraph' / f'{l}_{m}_bi_K9.npy', Tb)
    off = Tb.copy(); np.fill_diagonal(off, 0)
    peak = max(peak, off.max())
    print(f'{LANG_LABEL[l]} {METHOD_LABEL[m]:<12} {"ok":>5} {it:>7} {off.max():>13.4f}')

print(f'\npeak off-diagonal weight across all nine: {peak:.4f}')
print('paper §III-B: 26 iterations max, peak 0.40 -- compare notebook 07,')
print('where the same matrices without a floor reach 0.997.')

In [ ]:
# --- Table V: robust confusion pairs ---
from collections import Counter
cnt_all = Counter()
cnt_lang = {l: Counter() for l in LANGS}
for (l, m), Tb in BI_PAR.items():
    for i in range(K):
        for j in range(i + 1, K):
            if max(Tb[i, j], Tb[j, i]) >= THRESH:
                pair = (COMMON[i], COMMON[j])
                cnt_all[pair] += 1
                cnt_lang[l][pair] += 1

robust = pd.DataFrame([{'Tech_A': ABBR[a], 'Tech_B': ABBR[b], 'Cfg': f'{c}/9',
                        'EN': cnt_lang['en'][(a, b)], 'PL': cnt_lang['pl'][(a, b)],
                        'RU': cnt_lang['ru'][(a, b)]}
                       for (a, b), c in cnt_all.most_common() if c >= 3])
robust.to_csv(RESULTS / 'table5_robust_pairs.csv', index=False)
print(f'=== robust pairs (>= 3/9 configs, bi-norm weight >= {THRESH}) ===')
print(robust.to_string(index=False))

all3 = robust[(robust.EN > 0) & (robust.PL > 0) & (robust.RU > 0)]
print(f'\npresent in all three languages: {len(all3)}/{len(robust)}')
print('  ' + ', '.join(f'{r.Tech_A}<->{r.Tech_B}' for r in all3.itertuples()))
print('paper §IV-B: three of eight -- FW<->Sl, LL<->NCL, Sl<->Rep')

pathos = sum(1 for r in robust.itertuples()
             if TECH2SUPER[[t for t in COMMON if ABBR[t] == r.Tech_A][0]] == 'Pathos'
             and TECH2SUPER[[t for t in COMMON if ABBR[t] == r.Tech_B][0]] == 'Pathos')
print(f'\nwithin-Pathos: {pathos}/{len(robust)} against {6*5//2}/{K*(K-1)//2} '
      f'pairs available -- i.e. at chance (paper Limitations)')